# 056 — Graph Neural Networks

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** h₀' = media(0,1,2,3) = **1.5**; h₁' = media(1,0) = **0.5**;
h₂' = media(2,0) = **1.0**; h₃' = media(3,0) = **1.5**. El centro absorbe información
de todos en una capa; las hojas solo del centro.

**Ejercicio 2.** Sobre (1.5, 0.5, 1.0, 1.5): h₀'' = media de los 4 = **1.125**;
h₁'' = media(0.5, 1.5) = **1.0**; h₂'' = media(1.0, 1.5) = **1.25**;
h₃'' = media(1.5, 1.5) = **1.5**. Rangos: original 3 → capa 1: 1.0 → capa 2: 0.5.
Cada capa de promediado contrae el rango: over-smoothing medible en dos pasos.

**Ejercicio 3.** El nodo 6 está a distancia 5 del nodo 1 y cada capa propaga la
información un salto: hacen falta **5 capas** — pero el ejercicio 2 muestra que a esa
profundidad el promedio ya habrá difuminado gran parte de la señal (el dilema central
del diseño de GNN).

**Ejercicio 4.** El resultado por *nodo físico* es idéntico: la media no depende del
orden de los vecinos ni de la numeración. Esa es la invarianza a permutaciones, el
requisito que un MLP sobre la matriz de adyacencia no cumple.


In [ ]:
result = run_lab("neural", seed=56)
assert result["kind"] == "neural"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
vecinos = {0: [0, 1, 2, 3], 1: [1, 0], 2: [2, 0], 3: [3, 0]}
h = [0.0, 1.0, 2.0, 3.0]

def capa_media(h, vecinos):
    return [sum(h[u] for u in vecinos[v]) / len(vecinos[v]) for v in sorted(vecinos)]

h1 = capa_media(h, vecinos)
h2 = capa_media(h1, vecinos)
rango = lambda v: max(v) - min(v)
print("capa 1:", h1, "| capa 2:", h2)
print("rangos:", rango(h), "→", rango(h1), "→", rango(h2))
assert h1 == [1.5, 0.5, 1.0, 1.5]
assert h2 == [1.125, 1.0, 1.25, 1.5]

# Ejercicio 4: renumerar (intercambiar nodos 1 y 3) no cambia el resultado físico
vecinos_b = {0: [0, 3, 2, 1], 1: [1, 0], 2: [2, 0], 3: [3, 0]}
assert capa_media(h, vecinos_b) == h1
print("invarianza a permutaciones verificada")


## Reflexión

1. ¿Por qué la agregación debe ser invariante a permutaciones y qué descarta eso (p. ej. concatenar vecinos en orden)?
2. En el ejemplo del grafo línea, ¿cuántas capas necesita el nodo 1 para recibir información de un nodo a distancia 5, y qué le pasa mientras tanto al rango de los valores?
3. ¿En qué sentido preciso es la auto-atención del Transformer un caso particular de GNN, y qué aporta el grafo cuando sí existe?
